# monohunter — quickstart (zero install)

Hunt single long-period **mono-transits** in public TESS light curves, right in your browser — no Python setup.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rinkia/monohunter/blob/main/notebooks/monohunter_quickstart.ipynb)

Run each cell in order (Shift+Enter). It installs monohunter, recovers a known single transit to prove it works, then lets you scan any star you like.

## 1. Install

In [ ]:
%pip install -q monohunter

## 2. Prove it works — recover a known mono-transit

TOI-2180 b (TIC 298663873) is the canonical single-transit planet: one ~24 h dip in Sector 19. This should detect it and flag it as a known TESS Object of Interest.

In [ ]:
from monohunter.pipeline import run_target
from IPython.display import Image, display

records = run_target(298663873, sectors=[19], outdir="out", make_plots=True)
for r in records:
    print(f"S{r.sector}: depth={r.depth_ppt:.2f}ppt  dur={r.duration_hr:.0f}h  SNR={r.snr:.0f}"
          f"  {'known ' + r.known_toi_id if r.known_toi_match else 'NOT a known TOI'}")
    if r.plot_path:
        display(Image(r.plot_path))

**Always look at the plot.** A real transit is a coherent dip centered on the red line, on a flat baseline — not a sector-edge ramp, a data gap, or a single bad point.

## 3. Scan your own target

Change the TIC id below. Leave `sectors=None` to search every available sector (slower).

In [ ]:
TIC = 298663873   # <- put any TESS Input Catalog id here
SECTORS = None    # e.g. [19] to restrict, or None for all

records = run_target(TIC, sectors=SECTORS, outdir="out", make_plots=True)
if not records:
    print(f"No candidates for TIC {TIC} (nothing above the SNR threshold).")
for r in records:
    flags = []
    if r.known_toi_match: flags.append(f"known {r.known_toi_id}")
    if r.likely_eb: flags.append("likely eclipsing binary")
    if r.period_constrained and r.p_best_d:
        flags.append(f"P~{r.p_best_d:.0f}d ({r.p_lo_d:.0f}-{r.p_hi_d:.0f}d)")
    print(f"S{r.sector}: depth={r.depth_ppt:.2f}ppt  dur={r.duration_hr:.0f}h  SNR={r.snr:.0f}"
          f"  {' | '.join(flags) if flags else ''}")
    if r.plot_path:
        display(Image(r.plot_path))

## 4. Found something?

A candidate is **not** a discovery — it means a human thinks the dip is real. If the plot shows a clean, isolated transit on a flat baseline and it is **not** a known TOI:

- Re-run with a different detrend window (`run_target(TIC, window_length=5.0, ...)`); a real dip survives, an artifact moves or vanishes.
- Check it against the [community leaderboard](https://rinkia.github.io/monohunter/) and submit it — see the repo's **Contributing** section (there is a one-click issue form for non-coders).

Full docs: https://github.com/Rinkia/monohunter